In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, DateType
from delta.tables import DeltaTable

## CUSTOMERS CLEANING

In [0]:
df_cust = spark.table("resume_project.bronze_layer.bronze_customers")

display(df_cust)

customer_id,name,email,location,signup_date,_rescued_data,bronze_ingest_ts,_source_file
1,Madison Smith,williamsteresa@calhoun-fisher.com,New Darryl,2024-04-10,null,2026-04-06T18:10:16.895Z,/Volumes/resume_project/bronze_layer/bronze_volume/customers/dim_customers.csv
2,Kyle Brown,schroedermarco@hotmail.com,Lake Anthony,2023-05-26,null,2026-04-06T18:10:16.895Z,/Volumes/resume_project/bronze_layer/bronze_volume/customers/dim_customers.csv
3,Alexander Gonzalez,fgriffin@yahoo.com,Port Joseph,2022-09-09,null,2026-04-06T18:10:16.895Z,/Volumes/resume_project/bronze_layer/bronze_volume/customers/dim_customers.csv
4,Joseph Potter,paulsweeney@hotmail.com,Amandafort,2024-03-24,null,2026-04-06T18:10:16.895Z,/Volumes/resume_project/bronze_layer/bronze_volume/customers/dim_customers.csv
5,Joshua King,riverssheryl@george.com,Port Paulbury,2023-06-14,null,2026-04-06T18:10:16.895Z,/Volumes/resume_project/bronze_layer/bronze_volume/customers/dim_customers.csv
6,Spencer Pugh,jacob61@singh-johnson.net,Duranfort,2024-03-03,null,2026-04-06T18:10:16.895Z,/Volumes/resume_project/bronze_layer/bronze_volume/customers/dim_customers.csv
7,Kathryn Gordon,brookstina@yahoo.com,Lake Michael,2022-08-30,null,2026-04-06T18:10:16.895Z,/Volumes/resume_project/bronze_layer/bronze_volume/customers/dim_customers.csv
8,Greg Weaver,davidlewis@gmail.com,New Joseph,2024-04-06,null,2026-04-06T18:10:16.895Z,/Volumes/resume_project/bronze_layer/bronze_volume/customers/dim_customers.csv
9,Elizabeth Villanueva,richard84@gmail.com,West Ashleychester,2025-04-08,null,2026-04-06T18:10:16.895Z,/Volumes/resume_project/bronze_layer/bronze_volume/customers/dim_customers.csv
10,Darlene Morris,gregory16@gmail.com,Darrenmouth,2023-06-06,null,2026-04-06T18:10:16.895Z,/Volumes/resume_project/bronze_layer/bronze_volume/customers/dim_customers.csv


In [0]:
customers_clean = (
    df_cust
    .dropDuplicates(["customer_id"])
    .withColumn("name", F.lower(F.trim(F.col("name"))))
    .withColumn("email", F.lower(F.trim(F.col("email"))))
    .withColumn("signup_date", F.to_date("signup_date"))
    .withColumn("location", F.lower(F.trim(F.col("location"))))
    .withColumn("email_domain",F.split(F.split(F.col("email"), "@")[1], "\\.")[0])
)

In [0]:
display(customers_clean)

customer_id,name,email,location,signup_date,_rescued_data,bronze_ingest_ts,_source_file,email_domain
1,madison smith,williamsteresa@calhoun-fisher.com,new darryl,2024-04-10,null,2026-04-06T18:10:16.895Z,/Volumes/resume_project/bronze_layer/bronze_volume/customers/dim_customers.csv,calhoun-fisher
2,kyle brown,schroedermarco@hotmail.com,lake anthony,2023-05-26,null,2026-04-06T18:10:16.895Z,/Volumes/resume_project/bronze_layer/bronze_volume/customers/dim_customers.csv,hotmail
3,alexander gonzalez,fgriffin@yahoo.com,port joseph,2022-09-09,null,2026-04-06T18:10:16.895Z,/Volumes/resume_project/bronze_layer/bronze_volume/customers/dim_customers.csv,yahoo
4,joseph potter,paulsweeney@hotmail.com,amandafort,2024-03-24,null,2026-04-06T18:10:16.895Z,/Volumes/resume_project/bronze_layer/bronze_volume/customers/dim_customers.csv,hotmail
5,joshua king,riverssheryl@george.com,port paulbury,2023-06-14,null,2026-04-06T18:10:16.895Z,/Volumes/resume_project/bronze_layer/bronze_volume/customers/dim_customers.csv,george
6,spencer pugh,jacob61@singh-johnson.net,duranfort,2024-03-03,null,2026-04-06T18:10:16.895Z,/Volumes/resume_project/bronze_layer/bronze_volume/customers/dim_customers.csv,singh-johnson
7,kathryn gordon,brookstina@yahoo.com,lake michael,2022-08-30,null,2026-04-06T18:10:16.895Z,/Volumes/resume_project/bronze_layer/bronze_volume/customers/dim_customers.csv,yahoo
8,greg weaver,davidlewis@gmail.com,new joseph,2024-04-06,null,2026-04-06T18:10:16.895Z,/Volumes/resume_project/bronze_layer/bronze_volume/customers/dim_customers.csv,gmail
9,elizabeth villanueva,richard84@gmail.com,west ashleychester,2025-04-08,null,2026-04-06T18:10:16.895Z,/Volumes/resume_project/bronze_layer/bronze_volume/customers/dim_customers.csv,gmail
10,darlene morris,gregory16@gmail.com,darrenmouth,2023-06-06,null,2026-04-06T18:10:16.895Z,/Volumes/resume_project/bronze_layer/bronze_volume/customers/dim_customers.csv,gmail


In [0]:
target_table = "resume_project.silver_layer.customers_enr"

if spark.catalog.tableExists(target_table):
    dlt_obj = DeltaTable.forName(spark, target_table)

    (
        dlt_obj.alias("trg")
        .merge(
            customers_clean.alias("src"),
            "trg.customer_id = src.customer_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    (
        customers_clean.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )

## PRODUCTS CLEANING

In [0]:
df_prod = spark.table("resume_project.bronze_layer.bronze_products")
display(df_prod)

product_id,product_name,category,price,_rescued_data,bronze_ingest_ts,_source_file
1,Society Rest,Toys,190.4,null,2026-04-06T18:10:19.561Z,/Volumes/resume_project/bronze_layer/bronze_volume/products/dim_products.csv
2,Front Left,Toys,475.6,null,2026-04-06T18:10:19.561Z,/Volumes/resume_project/bronze_layer/bronze_volume/products/dim_products.csv
3,May Likely,Clothing,367.34,null,2026-04-06T18:10:19.561Z,/Volumes/resume_project/bronze_layer/bronze_volume/products/dim_products.csv
4,Gas Medical,Books,301.34,null,2026-04-06T18:10:19.561Z,/Volumes/resume_project/bronze_layer/bronze_volume/products/dim_products.csv
5,No West,Books,82.23,null,2026-04-06T18:10:19.561Z,/Volumes/resume_project/bronze_layer/bronze_volume/products/dim_products.csv
6,Form Rise,Electronics,82.22,null,2026-04-06T18:10:19.561Z,/Volumes/resume_project/bronze_layer/bronze_volume/products/dim_products.csv
7,Today Want,Clothing,33.75,null,2026-04-06T18:10:19.561Z,/Volumes/resume_project/bronze_layer/bronze_volume/products/dim_products.csv
8,Same All,Books,433.76,null,2026-04-06T18:10:19.561Z,/Volumes/resume_project/bronze_layer/bronze_volume/products/dim_products.csv
9,Brother Because,Clothing,302.55,null,2026-04-06T18:10:19.561Z,/Volumes/resume_project/bronze_layer/bronze_volume/products/dim_products.csv
10,Job Capital,Toys,355.5,null,2026-04-06T18:10:19.561Z,/Volumes/resume_project/bronze_layer/bronze_volume/products/dim_products.csv


In [0]:
products_clean = (
    df_prod
    .dropDuplicates(["product_id"])
    .withColumn("product_name", F.trim(F.col("product_name")))
    .withColumn("category", F.initcap(F.trim(F.col("category"))))
    .withColumn("price", F.col("price").cast(DoubleType()))
    .filter(F.col("price") >= 0)
)

In [0]:
display(products_clean)

product_id,product_name,category,price,_rescued_data,bronze_ingest_ts,_source_file
1,Society Rest,Toys,190.4,null,2026-04-06T18:10:19.561Z,/Volumes/resume_project/bronze_layer/bronze_volume/products/dim_products.csv
2,Front Left,Toys,475.6,null,2026-04-06T18:10:19.561Z,/Volumes/resume_project/bronze_layer/bronze_volume/products/dim_products.csv
3,May Likely,Clothing,367.34,null,2026-04-06T18:10:19.561Z,/Volumes/resume_project/bronze_layer/bronze_volume/products/dim_products.csv
4,Gas Medical,Books,301.34,null,2026-04-06T18:10:19.561Z,/Volumes/resume_project/bronze_layer/bronze_volume/products/dim_products.csv
5,No West,Books,82.23,null,2026-04-06T18:10:19.561Z,/Volumes/resume_project/bronze_layer/bronze_volume/products/dim_products.csv
6,Form Rise,Electronics,82.22,null,2026-04-06T18:10:19.561Z,/Volumes/resume_project/bronze_layer/bronze_volume/products/dim_products.csv
7,Today Want,Clothing,33.75,null,2026-04-06T18:10:19.561Z,/Volumes/resume_project/bronze_layer/bronze_volume/products/dim_products.csv
8,Same All,Books,433.76,null,2026-04-06T18:10:19.561Z,/Volumes/resume_project/bronze_layer/bronze_volume/products/dim_products.csv
9,Brother Because,Clothing,302.55,null,2026-04-06T18:10:19.561Z,/Volumes/resume_project/bronze_layer/bronze_volume/products/dim_products.csv
10,Job Capital,Toys,355.5,null,2026-04-06T18:10:19.561Z,/Volumes/resume_project/bronze_layer/bronze_volume/products/dim_products.csv


In [0]:
target_table = "resume_project.silver_layer.products_enr"

if spark.catalog.tableExists(target_table):
    dlt_obj = DeltaTable.forName(spark, target_table)

    (
        dlt_obj.alias("trg")
        .merge(
            products_clean.alias("src"),
            "trg.product_id = src.product_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    (
        products_clean.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )

## SALES CLEANING

In [0]:
df_sales = spark.table("resume_project.bronze_layer.bronze_sales")
display(df_sales)

sales_id,customer_id,product_id,store_id,quantity,discount,date,total_amount,_rescued_data,bronze_ingest_ts,_source_file
1,63,38,12,3,29.24,2024-09-21,113.25,null,2026-04-06T18:10:24.172Z,/Volumes/resume_project/bronze_layer/bronze_volume/sales/fact_sales.csv
2,96,97,14,4,1.57,2025-03-02,1038.44,null,2026-04-06T18:10:24.172Z,/Volumes/resume_project/bronze_layer/bronze_volume/sales/fact_sales.csv
3,52,23,14,4,20.47,2024-09-22,475.94,null,2026-04-06T18:10:24.172Z,/Volumes/resume_project/bronze_layer/bronze_volume/sales/fact_sales.csv
4,96,63,2,4,14.04,2024-08-22,1427.73,null,2026-04-06T18:10:24.172Z,/Volumes/resume_project/bronze_layer/bronze_volume/sales/fact_sales.csv
5,132,15,5,3,17.38,2024-11-11,235.47,null,2026-04-06T18:10:24.172Z,/Volumes/resume_project/bronze_layer/bronze_volume/sales/fact_sales.csv
6,151,97,5,3,2.56,2025-02-14,770.99,null,2026-04-06T18:10:24.172Z,/Volumes/resume_project/bronze_layer/bronze_volume/sales/fact_sales.csv
7,143,25,6,2,1.29,2025-06-01,455.55,null,2026-04-06T18:10:24.172Z,/Volumes/resume_project/bronze_layer/bronze_volume/sales/fact_sales.csv
8,171,17,13,1,20.01,2024-10-16,124.46,null,2026-04-06T18:10:24.172Z,/Volumes/resume_project/bronze_layer/bronze_volume/sales/fact_sales.csv
9,29,97,9,4,24.7,2025-03-28,794.42,null,2026-04-06T18:10:24.172Z,/Volumes/resume_project/bronze_layer/bronze_volume/sales/fact_sales.csv
10,36,66,15,4,23.46,2024-11-15,837.75,null,2026-04-06T18:10:24.172Z,/Volumes/resume_project/bronze_layer/bronze_volume/sales/fact_sales.csv


In [0]:
sales_clean = (
    df_sales
    .dropDuplicates()
    .withColumn("date", F.to_date("date"))
    .withColumn("quantity", F.col("quantity").cast(IntegerType()))
    .withColumn("discount", F.col("discount").cast(DoubleType()))
    .withColumn("total_amount", F.col("total_amount").cast(DoubleType()))
    .filter(F.col("customer_id").isNotNull())
    .filter(F.col("product_id").isNotNull())
    .filter(F.col("store_id").isNotNull())
    .filter(F.col("quantity") > 0)
    .filter(F.col("total_amount") >= 0)
)

In [0]:
display(sales_clean)

sales_id,customer_id,product_id,store_id,quantity,discount,date,total_amount,_rescued_data,bronze_ingest_ts,_source_file
68,81,27,5,1,19.53,2025-06-12,83.56,null,2026-04-06T18:10:24.172Z,/Volumes/resume_project/bronze_layer/bronze_volume/sales/fact_sales.csv
73,54,38,14,2,21.77,2025-03-12,83.47,null,2026-04-06T18:10:24.172Z,/Volumes/resume_project/bronze_layer/bronze_volume/sales/fact_sales.csv
84,68,82,18,1,24.3,2025-04-06,237.34,null,2026-04-06T18:10:24.172Z,/Volumes/resume_project/bronze_layer/bronze_volume/sales/fact_sales.csv
110,148,36,8,4,14.44,2025-01-06,1386.62,null,2026-04-06T18:10:24.172Z,/Volumes/resume_project/bronze_layer/bronze_volume/sales/fact_sales.csv
120,129,21,14,3,14.95,2025-02-10,785.53,null,2026-04-06T18:10:24.172Z,/Volumes/resume_project/bronze_layer/bronze_volume/sales/fact_sales.csv
130,90,35,16,1,0.43,2024-08-08,480.91,null,2026-04-06T18:10:24.172Z,/Volumes/resume_project/bronze_layer/bronze_volume/sales/fact_sales.csv
137,140,7,2,3,29.01,2025-05-01,71.88,null,2026-04-06T18:10:24.172Z,/Volumes/resume_project/bronze_layer/bronze_volume/sales/fact_sales.csv
155,200,29,12,4,24.24,2025-06-06,903.82,null,2026-04-06T18:10:24.172Z,/Volumes/resume_project/bronze_layer/bronze_volume/sales/fact_sales.csv
157,166,22,15,1,15.44,2024-09-12,62.62,null,2026-04-06T18:10:24.172Z,/Volumes/resume_project/bronze_layer/bronze_volume/sales/fact_sales.csv
163,93,81,12,1,0.01,2024-12-02,432.2,null,2026-04-06T18:10:24.172Z,/Volumes/resume_project/bronze_layer/bronze_volume/sales/fact_sales.csv


In [0]:
target_table = "resume_project.silver_layer.sales_enr"

if spark.catalog.tableExists(target_table):
    dlt_obj = DeltaTable.forName(spark, target_table)

    (
        dlt_obj.alias("trg")
        .merge(
            sales_clean.alias("src"),
            "trg.sales_id = src.sales_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    (
        sales_clean.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )

## STORES CLEANING

In [0]:
df_stores = spark.table("resume_project.bronze_layer.bronze_stores")
display(df_stores)

store_id,store_name,region,_rescued_data,bronze_ingest_ts,_source_file
1,Store_1,North,null,2026-04-06T18:10:27.784Z,/Volumes/resume_project/bronze_layer/bronze_volume/stores/dim_stores.csv
2,Store_2,West,null,2026-04-06T18:10:27.784Z,/Volumes/resume_project/bronze_layer/bronze_volume/stores/dim_stores.csv
3,Store_3,West,null,2026-04-06T18:10:27.784Z,/Volumes/resume_project/bronze_layer/bronze_volume/stores/dim_stores.csv
4,Store_4,South,null,2026-04-06T18:10:27.784Z,/Volumes/resume_project/bronze_layer/bronze_volume/stores/dim_stores.csv
5,Store_5,East,null,2026-04-06T18:10:27.784Z,/Volumes/resume_project/bronze_layer/bronze_volume/stores/dim_stores.csv
6,Store_6,South,null,2026-04-06T18:10:27.784Z,/Volumes/resume_project/bronze_layer/bronze_volume/stores/dim_stores.csv
7,Store_7,South,null,2026-04-06T18:10:27.784Z,/Volumes/resume_project/bronze_layer/bronze_volume/stores/dim_stores.csv
8,Store_8,West,null,2026-04-06T18:10:27.784Z,/Volumes/resume_project/bronze_layer/bronze_volume/stores/dim_stores.csv
9,Store_9,South,null,2026-04-06T18:10:27.784Z,/Volumes/resume_project/bronze_layer/bronze_volume/stores/dim_stores.csv
10,Store_10,East,null,2026-04-06T18:10:27.784Z,/Volumes/resume_project/bronze_layer/bronze_volume/stores/dim_stores.csv


In [0]:
stores_clean = (
    df_stores
    .dropDuplicates(["store_id"])
    .withColumn("store_name", F.trim(F.col("store_name")))
    .withColumn("region", F.upper(F.trim(F.col("region"))))
)

In [0]:
display(stores_clean)

store_id,store_name,region,_rescued_data,bronze_ingest_ts,_source_file
1,Store_1,NORTH,null,2026-04-06T18:10:27.784Z,/Volumes/resume_project/bronze_layer/bronze_volume/stores/dim_stores.csv
2,Store_2,WEST,null,2026-04-06T18:10:27.784Z,/Volumes/resume_project/bronze_layer/bronze_volume/stores/dim_stores.csv
3,Store_3,WEST,null,2026-04-06T18:10:27.784Z,/Volumes/resume_project/bronze_layer/bronze_volume/stores/dim_stores.csv
4,Store_4,SOUTH,null,2026-04-06T18:10:27.784Z,/Volumes/resume_project/bronze_layer/bronze_volume/stores/dim_stores.csv
5,Store_5,EAST,null,2026-04-06T18:10:27.784Z,/Volumes/resume_project/bronze_layer/bronze_volume/stores/dim_stores.csv
6,Store_6,SOUTH,null,2026-04-06T18:10:27.784Z,/Volumes/resume_project/bronze_layer/bronze_volume/stores/dim_stores.csv
7,Store_7,SOUTH,null,2026-04-06T18:10:27.784Z,/Volumes/resume_project/bronze_layer/bronze_volume/stores/dim_stores.csv
8,Store_8,WEST,null,2026-04-06T18:10:27.784Z,/Volumes/resume_project/bronze_layer/bronze_volume/stores/dim_stores.csv
9,Store_9,SOUTH,null,2026-04-06T18:10:27.784Z,/Volumes/resume_project/bronze_layer/bronze_volume/stores/dim_stores.csv
10,Store_10,EAST,null,2026-04-06T18:10:27.784Z,/Volumes/resume_project/bronze_layer/bronze_volume/stores/dim_stores.csv


In [0]:
target_table = "resume_project.silver_layer.stores_enr"

if spark.catalog.tableExists(target_table):
    dlt_obj = DeltaTable.forName(spark, target_table)

    (
        dlt_obj.alias("trg")
        .merge(
            stores_clean.alias("src"),
            "trg.store_id = src.store_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    (
        stores_clean.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )